# 6 — prompt iteration

Every book so far has run on the same system prompt: one short paragraph,
written in book 3 and never looked at again. Whatever the agent got right or
wrong, part of the reason was that paragraph. This book writes three versions
of it, sends the same five requests through each, and compares what each
version got right.

**Five requests with known answers.** To compare prompts you need requests
whose right answer you already know, so you can tell a good reply from a
plausible-sounding one. Each of these five has one you can check against
`data/hotels.json` and `data/policies/`:

| # | asks for | right answer |
| --- | --- | --- |
| 1 | Lisbon, under €150 a night, breakfast | `lis-002` (€125) or `lis-004` (€140) |
| 2 | Madrid, under €90 a night | nothing — the cheapest Madrid hotel is €100 |
| 3 | Seville, 10–12 November | nothing — no hotel is open in November |
| 4 | Porto, under €160, with a 30 kg dog | `por-004` or `por-005`; the pet fee is €25 per stay, and 30 kg is over the 20 kg limit, so the hotel has to agree first |
| 5 | "somewhere nice in Porto" | no proposal yet — a question back about dates and budget |

None of them asks to book, so the approval gate from book 5 never fires here.

In [1]:
from hotelbot.config import configure_tracing
from hotelbot.agent import build_agent, run
from hotelbot.prompts import get_prompt
from hotelbot.samples import REQUESTS

configure_tracing(book=6)

for i, request in enumerate(REQUESTS, 1):
    print(f"{i}. {request}")

1. Two nights in Lisbon, 2026-10-10 to 2026-10-12, under €150 a night, with breakfast included.
2. Three nights in Madrid, 2026-10-05 to 2026-10-08, under €90 a night.
3. Two nights in Seville, 2026-11-10 to 2026-11-12, under €150 a night.
4. Two nights in Porto, 2026-10-16 to 2026-10-18, under €160 a night. I'm bringing my dog, she's 30 kg — what will that cost me, and which hotel would you pick?
5. Somewhere nice in Porto.


**One row per request.** `run_all(version)` builds the agent with one prompt
version and sends it the five requests, each on a new thread so no request
sees another's conversation. `show` prints a row for each: first the tools
the agent called, in order, then its final answer. The tools line is how you
tell a checked answer from a guessed one — a pet fee quoted with no
`lookup_policy` call came from the model's own head, not from
`data/policies/pets.md`. The final-answer tools (`BookingProposal`,
`PlainAnswer`) are left out of that line, since every run ends with one.

In [2]:
from langchain_core.messages import AIMessage

ANSWER_TOOLS = {"BookingProposal", "PlainAnswer"}


def run_all(version):
    agent = build_agent(prompt_version=version)
    return [run(agent, request) for request in REQUESTS]


def tools_called(result):
    return [
        call["name"]
        for message in result.messages
        if isinstance(message, AIMessage)
        for call in message.tool_calls
        if call["name"] not in ANSWER_TOOLS
    ]


def show(results):
    for i, result in enumerate(results, 1):
        print(f"{i}. tools: {', '.join(tools_called(result)) or 'none'}")
        print(f"   {result.structured_response!r}")
        print()

**What v1 is told.** Here is the whole prompt every earlier book used. Read
it for what it leaves out as much as for what it says: it is the only
instruction the model gets beyond the tool descriptions.

In [3]:
print(get_prompt("v1"))

You are a hotel booking assistant for Lisbon, Porto, Madrid, and Seville. Ask only for what you need to search: city, dates, and budget. Keep replies short. Today is 2026-09-13. Produce dates in ISO format (YYYY-MM-DD).


**What v1 does.** Now the five requests through v1. Predict before running:
using the right answers in the table above, which of the five will v1 get
wrong, and in what way? Then compare each row against the table — check the
hotel id, the price against the budget, the dates, and whether the tools
line shows the agent actually looked.

In [4]:
v1_results = run_all("v1")
show(v1_results)

1. tools: search_hotels, check_availability
   BookingProposal(hotel_id='lis-004', check_in='2026-10-10', check_out='2026-10-12', nights=2, total=280.0, why="Casa do Castelo in Alfama, €140/night with breakfast included, rated 4.6, available for your dates. (Alfama Loft also qualifies at €125/night if you'd prefer lower cost/rating trade-off.)")

2. tools: search_hotels
   PlainAnswer(text="I couldn't find any hotels in Madrid under €90/night. Would you like me to raise the budget or try different dates?")

3. tools: search_hotels
   PlainAnswer(text="Unfortunately none of the Seville hotels under €150/night show availability for Nov 10–12, 2026 — they're all listed as available only Oct 1–31 and Dec 1–20. Would you like me to check a different date range or a higher budget?")

4. tools: search_hotels, lookup_policy, check_availability
   BookingProposal(hotel_id='por-004', check_in='2026-10-16', check_out='2026-10-18', nights=2, total=297.0, why="Foz do Douro Palace (€136/night, pets 

**Rules.** v1 says who the assistant is and what to ask for, but nothing about
when a hotel is good enough to propose. Whether to check the dates, whether
the budget is a hard limit, what to do when nothing fits, where a policy
answer should come from — all of that was left to the model. v2 is v1 with
four rules appended: plain sentences, one per behaviour. Because v2 starts
with v1's exact text, cutting v1 off the front leaves only what was added.

In [5]:
v1, v2 = get_prompt("v1"), get_prompt("v2")
assert v2.startswith(v1)
print(v2.removeprefix(v1))



Rules:
- Before proposing a hotel, call check_availability for it on the guest's dates. Propose it only if the result says it is available, and use the nights and total from that result.
- Never propose a hotel whose price per night is over the guest's stated budget.
- If no hotel meets every requirement, say so plainly and propose nothing. Do not offer a near miss as a proposal.
- Answer any question about hotel policy (pets, cancellation, payment, check-in, accessibility) only from what lookup_policy returns, never from general knowledge.


**What v2 does.** The same five requests, now under v2. Predict before
running: which rows from v1 will change, and which will stay the same even
though a rule now covers them?

In [6]:
v2_results = run_all("v2")
show(v2_results)

1. tools: search_hotels, check_availability
   BookingProposal(hotel_id='lis-004', check_in='2026-10-10', check_out='2026-10-12', nights=2, total=280.0, why='Casa do Castelo (Alfama) has breakfast, €140/night (under €150), rated 4.6, and is available for your dates — total €280 for 2 nights.')

2. tools: search_hotels, search_hotels, search_hotels
   PlainAnswer(text="Sorry, I couldn't find any Madrid hotel under €90/night — the cheapest option (Malasana House) is €100/night. Want me to check a higher budget?")

3. tools: search_hotels
   PlainAnswer(text="Unfortunately, none of the Seville hotels under €150/night are available for Nov 10-12, 2026 — they're all only available in October or December this autumn. I can't propose a stay for those dates. Would you like different dates, or check availability in another city?")

4. tools: search_hotels, lookup_policy, check_availability
   PlainAnswer(text="Heads up: your dog is 30 kg, over the standard 20 kg pet limit, so she'd need the hot

**What the rules changed in the tool calls.** A rule only matters if the
agent behaves differently, and the clearest sign is in the tools it calls.
This prints, for each request, the tools called under v1 and under v2 one
above the other, then the kind of final answer — `BookingProposal` if it
proposed a hotel, `PlainAnswer` if it replied in words. Look for a row where
a rule changed what the agent did, and one where it changed nothing — both
tell you something about what the rules are worth. For any row that changed,
find the same request in the `hotelbot-book-6` project in LangSmith to see
the full exchange behind it.

In [7]:
def tools_and_answer(result):
    tools = ", ".join(tools_called(result)) or "none"
    return f"{tools} → {type(result.structured_response).__name__}"


for i, (old, new) in enumerate(zip(v1_results, v2_results), 1):
    print(f"{i}. v1: {tools_and_answer(old)}")
    print(f"   v2: {tools_and_answer(new)}")
    print()

1. v1: search_hotels, check_availability → BookingProposal
   v2: search_hotels, check_availability → BookingProposal

2. v1: search_hotels → PlainAnswer
   v2: search_hotels, search_hotels, search_hotels → PlainAnswer

3. v1: search_hotels → PlainAnswer
   v2: search_hotels → PlainAnswer

4. v1: search_hotels, lookup_policy, check_availability → BookingProposal
   v2: search_hotels, lookup_policy, check_availability → PlainAnswer

5. v1: search_hotels → PlainAnswer
   v2: none → PlainAnswer



**Examples.** A rule tells the model what to do in words; the model still has
to work out what following it looks like. The other way to steer it is to
show a finished case: a request, the tools called for it, and the exact
answer it should end with. Examples placed in the prompt like this are
called *few-shot* examples — "few" because there are only a handful, "shot"
meaning one worked case. v3 is v2 with two of them appended: one where a
hotel fits and comes with a caveat, one where nothing fits. Neither is one
of the five requests. As before, cutting v2 off the front leaves only what
v3 adds.

In [8]:
v3 = get_prompt("v3")
assert v3.startswith(v2)
print(v3.removeprefix(v2))



Two examples of a request and the answer it should get:

Example 1
Guest: Three nights in Seville, 2026-10-02 to 2026-10-05, under €110 a night, with a gym. I'll arrive around 23:00.
You call search_hotels, then check_availability for sev-001, then lookup_policy about late arrival.
You answer with a BookingProposal:
  hotel_id: sev-001
  check_in: 2026-10-02
  check_out: 2026-10-05
  nights: 3
  total: 240.0
  why: Santa Cruz House, €80 a night with a gym, available for all three nights. Arriving after 22:00 must be arranged with the hotel at least a day in advance.

Example 2
Guest: Two nights in Madrid, 2026-12-22 to 2026-12-24, under €200 a night.
You call search_hotels, then check_availability for each hotel under €200; none is available on those dates.
You answer with a PlainAnswer:
  text: Nothing in Madrid under €200 a night is available 22-24 December; the hotels there take bookings only until 20 December. Would earlier dates work?


**What v3 does.** The five requests a third time. Predict before running:
does showing an answer change anything that telling (v2) didn't? Rows 2 and 3
are the "nothing fits" cases, the same kind as example 2; row 4 is a hotel
with a caveat, the same kind as example 1.

In [9]:
v3_results = run_all("v3")
show(v3_results)

1. tools: search_hotels, check_availability, check_availability
   BookingProposal(hotel_id='lis-004', check_in='2026-10-10', check_out='2026-10-12', nights=2, total=280.0, why='Casa do Castelo, Alfama: €140/night with breakfast (plus gym, wifi), rated 4.6, available for both nights and under your €150 budget.')

2. tools: search_hotels, search_hotels, search_hotels
   PlainAnswer(text="I don't have any Madrid hotels under €90/night — the cheapest is Malasana House at €100. Would you like to raise your budget or try different dates?")

3. tools: search_hotels
   PlainAnswer(text="Unfortunately, none of Seville's hotels under €150 are available for November 10-12, 2026 — they're only bookable through October or starting again December 1. Would different dates work?")

4. tools: search_hotels, lookup_policy, check_availability
   BookingProposal(hotel_id='por-004', check_in='2026-10-16', check_out='2026-10-18', nights=2, total=272.0, why="Foz do Douro Palace, €136/night, pets allowed, av

**Three versions, one row each.** The same comparison as before, now with v3
added under each request. Read down a row to see how one request was handled
under each prompt; a line that never changes across all three is a
behaviour the prompt didn't touch.

In [10]:
for i, rows in enumerate(zip(v1_results, v2_results, v3_results), 1):
    for version, result in zip(["v1", "v2", "v3"], rows):
        label = f"{i}." if version == "v1" else "  "
        print(f"{label} {version}: {tools_and_answer(result)}")
    print()

1. v1: search_hotels, check_availability → BookingProposal
   v2: search_hotels, check_availability → BookingProposal
   v3: search_hotels, check_availability, check_availability → BookingProposal

2. v1: search_hotels → PlainAnswer
   v2: search_hotels, search_hotels, search_hotels → PlainAnswer
   v3: search_hotels, search_hotels, search_hotels → PlainAnswer

3. v1: search_hotels → PlainAnswer
   v2: search_hotels → PlainAnswer
   v3: search_hotels → PlainAnswer

4. v1: search_hotels, lookup_policy, check_availability → BookingProposal
   v2: search_hotels, lookup_policy, check_availability → PlainAnswer
   v3: search_hotels, lookup_policy, check_availability → BookingProposal

5. v1: search_hotels → PlainAnswer
   v2: none → PlainAnswer
   v3: none → PlainAnswer



**What code can check for you.** Part of the right-answer table is
mechanical. When the agent proposes a hotel, the catalogue can say whether
that hotel really is open on those dates, and what the stay really costs —
price per night times nights. `check_proposal` asks it and compares: a
`total` that differs from the catalogue's means the agent added something
(a fee, say) or got the sum wrong. For a `PlainAnswer` there is nothing to
look up; whether "nothing fits" or "what dates?" was the right reply is
yours to judge against the table at the top.

In [11]:
from hotelbot.catalogue import check_availability, get_hotel
from hotelbot.models import BookingProposal


def check_proposal(answer):
    if not isinstance(answer, BookingProposal):
        return "no proposal"
    found = check_availability(answer.hotel_id, answer.check_in, answer.check_out)
    if not found.ok:
        return f"{answer.hotel_id} NOT available: {found.reason}"
    price = get_hotel(answer.hotel_id).price_per_night
    if answer.total != found.total:
        return f"{answer.hotel_id} available, €{price:.0f}/night, but total {answer.total} ≠ catalogue {found.total}"
    return f"{answer.hotel_id} available, €{price:.0f}/night, total {answer.total} matches"


for i, rows in enumerate(zip(v1_results, v2_results, v3_results), 1):
    for version, result in zip(["v1", "v2", "v3"], rows):
        label = f"{i}." if version == "v1" else "  "
        print(f"{label} {version}: {check_proposal(result.structured_response)}")
    print()

1. v1: lis-004 available, €140/night, total 280.0 matches
   v2: lis-004 available, €140/night, total 280.0 matches
   v3: lis-004 available, €140/night, total 280.0 matches

2. v1: no proposal
   v2: no proposal
   v3: no proposal

3. v1: no proposal
   v2: no proposal
   v3: no proposal

4. v1: por-004 available, €136/night, but total 297.0 ≠ catalogue 272.0
   v2: no proposal
   v3: por-004 available, €136/night, total 272.0 matches

5. v1: no proposal
   v2: no proposal
   v3: no proposal



**Scoring by hand.** One mark per request per version — fifteen in all. A
reply earns ✓ if it matches the right answer in the table at the top *and*
everything it states is true: the hotel, the dates, the total, any policy it
quotes. Otherwise it gets a short word saying what went wrong — `total`,
`no proposal`, `over budget`, `guessed`. Use the rows from `show` and the
checks above, fill in `SCORES`, and run the cell. Each column's count of ✓ is
that version's score. These are your marks for your run: someone else's run
can differ, because the model doesn't answer the same way every time.

In [12]:
OK = "✓"

SCORES = {
    #   v1    v2    v3     ← replace each "?" with OK or a word
    1: ["?", "?", "?"],
    2: ["?", "?", "?"],
    3: ["?", "?", "?"],
    4: ["?", "?", "?"],
    5: ["?", "?", "?"],
}

print(f"{'':4}{'v1':16}{'v2':16}{'v3':16}")
for i, marks in SCORES.items():
    cells = [mark if mark in (OK, "?") else f"✗ {mark}" for mark in marks]
    print(f"{i:<4}" + "".join(f"{cell:16}" for cell in cells))
totals = [sum(marks[column] == OK for marks in SCORES.values()) for column in range(3)]
print(f"{'✓':4}" + "".join(f"{total}/5".ljust(16) for total in totals))

    v1              v2              v3              
1   ?               ?               ?               
2   ?               ?               ?               
3   ?               ?               ?               
4   ?               ?               ?               
5   ?               ?               ?               
✓   0/5             0/5             0/5             


**What a longer prompt costs.** The system prompt isn't sent once per
conversation: it goes out again with every model call, and one request takes
several calls — one per round of tool use, plus the final answer. Each
`AIMessage` carries `usage_metadata` (book 1), whose `input_tokens` is how
many tokens that call sent in. The first call of request 1 sends the same
tools and the same request under every version, so the only difference
between versions on that call is the prompt itself. Predict before running:
how many more input tokens does v3's first call send than v1's? Then the
total over all five requests — every call pays the difference again.

In [13]:
def input_tokens_per_call(result):
    return [
        message.usage_metadata["input_tokens"]
        for message in result.messages
        if isinstance(message, AIMessage) and message.usage_metadata
    ]


for version, results in [("v1", v1_results), ("v2", v2_results), ("v3", v3_results)]:
    first_call = input_tokens_per_call(results[0])[0]
    calls = sum(len(input_tokens_per_call(result)) for result in results)
    total = sum(sum(input_tokens_per_call(result)) for result in results)
    print(f"{version}: first call of request 1 = {first_call:>5} | all five: {calls:>2} calls, {total:>6} input tokens")

v1: first call of request 1 =  2513 | all five: 12 calls,  34911 input tokens
v2: first call of request 1 =  2690 | all five: 13 calls,  40140 input tokens
v3: first call of request 1 =  3065 | all five: 13 calls,  45204 input tokens


**Storing a prompt by name.** So far the three versions live in
`prompts.py`, and changing one means editing code. LangSmith also keeps a
*prompt hub*: a store where a prompt is saved under a name, and every save —
a *push* — becomes a new *commit*, identified by a *commit hash*, a string
of hex characters such as `3f9a1c2e…`. Nothing is overwritten: older commits
stay and can still be fetched. The hub stores a prompt *template* (a list of
messages, possibly with blanks to fill in), so the text is wrapped in a
`ChatPromptTemplate` holding one system message. `push_prompt` returns a link
to the commit in the LangSmith UI; `pull_prompt_commit` reads back the full
hash of the latest one.

In [14]:
from langchain_core.prompts import ChatPromptTemplate
from langsmith import Client

hub = Client()
PROMPT_NAME = "hotelbot-booking"


def push(text):
    url = hub.push_prompt(PROMPT_NAME, object=ChatPromptTemplate.from_messages([("system", text)]))
    commit = hub.pull_prompt_commit(PROMPT_NAME, skip_cache=True).commit_hash
    return url, commit


v3_url, v3_commit = push(v3)
print(v3_url)
print("commit:", v3_commit)

https://smith.langchain.com/prompts/hotelbot-booking/13eaafb5?organizationId=d590007d-a6a0-46e5-9a56-baddd100e9d2
commit: 13eaafb5ed0770fb4d83b3fdbef31411896a8455c07b236ff8d82d80863dab0a


**Pulling it back and running on it.** `get_prompt("hub:hotelbot-booking")`
fetches the latest commit from the hub and returns its text, so it can go
anywhere a local version name can — including `build_agent`. If the pulled
text equals v3 character for character, the agent can't tell the
difference. Request 4 runs on it below; compare the row with v3's.

In [15]:
pulled = get_prompt(f"hub:{PROMPT_NAME}")
print("same text as v3:", pulled == v3)

hub_agent = build_agent(prompt_version=f"hub:{PROMPT_NAME}")
show([run(hub_agent, REQUESTS[3])])

same text as v3: True
1. tools: search_hotels, lookup_policy, check_availability
   BookingProposal(hotel_id='por-004', check_in='2026-10-16', check_out='2026-10-18', nights=2, total=272.0, why="Foz do Douro Palace, €136/night, available for your dates and allows pets. Note: pets over 20 kg need the hotel's written advance sign-off (your dog is 30 kg) — contact them directly before booking. Pet fee is a flat €25 per stay, charged at check-in.")



**Older commits stay.** Now push v2 under the same name. It becomes the
latest commit, with a new hash — but v3's commit is still there, and
`"hub:<name>:<commit>"` fetches one particular commit instead of the latest.
Predict before running: after the push, which version does
`"hub:hotelbot-booking"` return, and which does
`"hub:hotelbot-booking:<v3's hash>"` return?

In [16]:
v2_url, v2_commit = push(v2)
print("v2 commit:", v2_commit)
print("v3 commit:", v3_commit)
print()
print("latest is v2:      ", get_prompt(f"hub:{PROMPT_NAME}") == v2)
print("v3's commit is v3: ", get_prompt(f"hub:{PROMPT_NAME}:{v3_commit}") == v3)

v2 commit: c829e8edf5ee778c7357a334afca9d7f2acabf574c536d0e0aa91ff7fd20065d
v3 commit: 13eaafb5ed0770fb4d83b3fdbef31411896a8455c07b236ff8d82d80863dab0a

latest is v2:       True
v3's commit is v3:  True


**What you can do now.** The system prompt is something you can measure
instead of something you write once and trust. You have three versions of
it, the same five requests through each, a check of every proposal against
the catalogue, a score per version, and the token cost of each. Your table
and the token counts answer the question this book asked: on this model,
how much did the rules and the examples buy, and was it worth sending the
extra tokens on every call? The prompt is also stored by name in the hub,
with every version kept by its commit hash, so a notebook can pull a
particular one instead of pasting text.

What none of this covers is a turn where something underneath goes wrong:
a tool that fails, or a policy document that has been tampered with to give
the model instructions. Book 7 breaks both on purpose and makes the agent
survive it.